# Notebook 6 – Probabilidad con datos meteorológicos



## 1. Cargar los datos

El archivo trae 9 líneas de encabezado (ciudad, agencia, URL) antes de los nombres de columna,
por eso se usa skiprows=9.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/notebook6/meteorología_2025.csv'



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df = pd.read_csv(RUTA, skiprows=9)

print('Filas:', len(df))
print('Columnas:', list(df.columns))
df.head()

Filas: 1191360
Columnas: ['date', 'id_station', 'id_parameter', 'valor', 'unit']


,date,id_station,id_parameter,valor,unit
0,2025-01-01 00:00:00,ACO,TMP,8.7,5
1,2025-01-01 00:00:00,ACO,RH,28.0,6
2,2025-01-01 00:00:00,ACO,WSP,2.4,3
3,2025-01-01 00:00:00,ACO,WDR,2.0,4
4,2025-01-01 00:00:00,AJM,TMP,9.6,5


## 2. Seleccionar la estación


In [ ]:
print('Parámetros:', df['id_parameter'].unique())
print('Estaciones:', df['id_station'].nunique())

Parámetros: ['TMP' 'RH' 'WSP' 'WDR']
Estaciones: 34


In [ ]:
ESTACION = 'MER'   # Merced

est = df[(df['id_station'] == ESTACION) & (df['id_parameter'].isin(['RH', 'WSP']))]

print('Registros de la estación', ESTACION, ':', len(est))

Registros de la estación MER : 17520


In [ ]:
datos = est.pivot_table(index='date', columns='id_parameter',
                        values='valor', aggfunc='first')

# Solo observaciones que tienen las dos mediciones
datos = datos.dropna(subset=['RH', 'WSP'])

print('Observaciones:', len(datos))
datos.head()

Observaciones: 8740


id_parameter,RH,WSP
date,,
2025-01-01 00:00:00,28.0,2.2
2025-01-01 01:00:00,29.0,2.4
2025-01-01 02:00:00,29.0,1.3
2025-01-01 03:00:00,30.0,0.8
2025-01-01 04:00:00,33.0,1.0


## 3. Definir los eventos

- Humedad alta: RH >= 70
- Viento tranquilo: WSP <= 1

In [ ]:
humedad_alta    = datos['RH'] >= 70
viento_tranquilo = datos['WSP'] <= 1

total    = len(datos)
n_hum    = humedad_alta.sum()                        # observaciones con humedad alta
n_vie    = viento_tranquilo.sum()                    # observaciones con viento tranquilo
n_ambos  = (humedad_alta & viento_tranquilo).sum()   # observaciones con las dos

print('Total de observaciones :', total)
print('Humedad alta           :', n_hum)
print('Viento tranquilo       :', n_vie)
print('Las dos a la vez       :', n_ambos)

Total de observaciones : 8740
Humedad alta           : 1943
Viento tranquilo       : 1501
Las dos a la vez       : 632


## 4. Cálculos

### a) Probabilidad de humedad alta

In [ ]:
P_hum = n_hum / total

print('P(humedad alta) =', n_hum, '/', total, '=', round(P_hum, 4))

P(humedad alta) = 1943 / 8740 = 0.2223


### b) Probabilidad de viento tranquilo

In [ ]:
P_vie = n_vie / total

print('P(viento tranquilo) =', n_vie, '/', total, '=', round(P_vie, 4))

P(viento tranquilo) = 1501 / 8740 = 0.1717


### c) P(humedad alta | viento tranquilo)

In [ ]:
P_hum_dado_vie = n_ambos / n_vie

print('P(humedad alta | viento tranquilo) =', n_ambos, '/', n_vie, '=', round(P_hum_dado_vie, 4))

P(humedad alta | viento tranquilo) = 632 / 1501 = 0.4211


### d) P(viento tranquilo | humedad alta)

In [ ]:
P_vie_dado_hum = n_ambos / n_hum

print('P(viento tranquilo | humedad alta) =', n_ambos, '/', n_hum, '=', round(P_vie_dado_hum, 4))

P(viento tranquilo | humedad alta) = 632 / 1943 = 0.3253


## Resultados

In [ ]:
print('Estación:', ESTACION, '|', total, 'observaciones\n')
print('a) P(humedad alta)                     =', round(P_hum, 4))
print('b) P(viento tranquilo)                 =', round(P_vie, 4))
print('c) P(humedad alta | viento tranquilo)  =', round(P_hum_dado_vie, 4))
print('d) P(viento tranquilo | humedad alta)  =', round(P_vie_dado_hum, 4))

Estación: MER | 8740 observaciones

a) P(humedad alta)                     = 0.2223
b) P(viento tranquilo)                 = 0.1717
c) P(humedad alta | viento tranquilo)  = 0.4211
d) P(viento tranquilo | humedad alta)  = 0.3253
